In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder , StandardScaler
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')
#load data
df = pd.read_csv(r'C:\Users\riyas\OneDrive\Desktop\Credit_Risk_project\data\train.csv')
# Select features
features = [
    'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY',
    'AMT_GOODS_PRICE', 'DAYS_BIRTH', 'DAYS_EMPLOYED',
    'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3',
    'NAME_CONTRACT_TYPE', 'NAME_INCOME_TYPE', 
    'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS',
    'REGION_POPULATION_RELATIVE', 'CNT_CHILDREN'
]

df_model = df[features + ['default']].copy()
df_model['DAYS_EMPLOYED'] = df_model['DAYS_EMPLOYED'].replace(365243, np.nan)

cat_cols = df_model.select_dtypes(include='object').columns
le = LabelEncoder()
for col in cat_cols:
    df_model[col] = df_model[col].fillna('Unknown')
    df_model[col] = le.fit_transform(df_model[col].astype(str))

num_cols = df_model.select_dtypes(include=np.number).columns
df_model[num_cols] = df_model[num_cols].fillna(df_model[num_cols].median())

X = df_model.drop('default', axis=1)
y = df_model['default']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

sm = SMOTE(random_state=42)
X_train_sm, y_train_sm = sm.fit_resample(X_train, y_train)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_sm)
X_test_scaled = scaler.transform(X_test)

lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_scaled, y_train_sm)

lr_probs = lr.predict_proba(X_test_scaled)[:, 1]
print("Model ready. Predictions generated.")

Model ready. Predictions generated.


In [2]:
# Select features
features = [
    'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY',
    'AMT_GOODS_PRICE', 'DAYS_BIRTH', 'DAYS_EMPLOYED',
    'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3',
    'NAME_CONTRACT_TYPE', 'NAME_INCOME_TYPE', 
    'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS',
    'REGION_POPULATION_RELATIVE', 'CNT_CHILDREN'
]

df_model = df[features + ['default']].copy()
df_model['DAYS_EMPLOYED'] = df_model['DAYS_EMPLOYED'].replace(365243, np.nan)

cat_cols = df_model.select_dtypes(include='object').columns
le = LabelEncoder()
for col in cat_cols:
    df_model[col] = df_model[col].fillna('Unknown')
    df_model[col] = le.fit_transform(df_model[col].astype(str))

num_cols = df_model.select_dtypes(include=np.number).columns
df_model[num_cols] = df_model[num_cols].fillna(df_model[num_cols].median())

X = df_model.drop('default', axis=1)
y = df_model['default']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

sm = SMOTE(random_state=42)
X_train_sm, y_train_sm = sm.fit_resample(X_train, y_train)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_sm)
X_test_scaled = scaler.transform(X_test)

lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_scaled, y_train_sm)

lr_probs = lr.predict_proba(X_test_scaled)[:, 1]
print("Model ready. Predictions generated.")

Model ready. Predictions generated.


In [3]:
# Build results DataFrame
df_results = X_test.copy()
df_results['DEFAULT_PROBABILITY'] = lr_probs
df_results['ACTUAL_DEFAULT'] = y_test.values

# Assign risk bands
def assign_risk_band(prob):
    if prob < 0.15:
        return 'GREEN'
    elif prob < 0.35:
        return 'AMBER'
    elif prob < 0.60:
        return 'RED'
    else:
        return 'BLACK'

df_results['RISK_BAND'] = df_results['DEFAULT_PROBABILITY'].apply(assign_risk_band)

# Band summary
band_summary = df_results.groupby('RISK_BAND').agg(
    count=('ACTUAL_DEFAULT', 'count'),
    actual_default_rate=('ACTUAL_DEFAULT', 'mean'),
    avg_default_prob=('DEFAULT_PROBABILITY', 'mean')
).round(3)

print(band_summary)

           count  actual_default_rate  avg_default_prob
RISK_BAND                                              
AMBER        593                0.032             0.248
BLACK        376                0.170             0.740
GREEN        278                0.025             0.091
RED          553                0.085             0.468


In [4]:
# Save band summary
band_summary.to_csv(r'C:\Users\riyas\OneDrive\Desktop\Credit_Risk_project\output\risk_band_summary.csv')
print("Saved.")

Saved.


In [5]:
band_summary.to_csv(r'C:\Users\riyas\OneDrive\Desktop\Credit_Risk_project\risk_band_summary.csv')
print("Saved.")

Saved.


In [6]:
# Isolate AMBER band non-defaulters
amber = df_results[df_results['RISK_BAND'] == 'AMBER'].copy()
amber_good = amber[amber['ACTUAL_DEFAULT'] == 0].copy()

print(f"Total AMBER applicants: {len(amber)}")
print(f"AMBER non-defaulters: {len(amber_good)}")
print(f"AMBER actual default rate: {amber['ACTUAL_DEFAULT'].mean():.2%}")

# Profile the non-defaulters
print("\nIncome profile of AMBER non-defaulters:")
print(amber_good['AMT_INCOME_TOTAL'].describe())

print("\nAge profile (years):")
print((abs(amber_good['DAYS_BIRTH']) / 365).describe())

print("\nLoan amount profile:")
print(amber_good['AMT_CREDIT'].describe())

Total AMBER applicants: 593
AMBER non-defaulters: 574
AMBER actual default rate: 3.20%

Income profile of AMBER non-defaulters:
count       574.000000
mean     170571.927700
std       92438.459408
min       40500.000000
25%      112500.000000
50%      148500.000000
75%      202500.000000
max      774000.000000
Name: AMT_INCOME_TOTAL, dtype: float64

Age profile (years):
count    574.000000
mean      46.392463
std       11.730581
min       21.109589
25%       36.464384
50%       46.242466
75%       57.174658
max       68.780822
Name: DAYS_BIRTH, dtype: float64

Loan amount profile:
count    5.740000e+02
mean     6.190591e+05
std      4.180172e+05
min      5.094000e+04
25%      2.700000e+05
50%      5.352930e+05
75%      8.403266e+05
max      2.463840e+06
Name: AMT_CREDIT, dtype: float64


In [7]:
# Calculate lost interest income
avg_loan = amber_good['AMT_CREDIT'].mean()
interest_rate = 0.15  # 15% annual interest rate
avg_loan_tenure = 3   # 3 years average

lost_interest_per_applicant = avg_loan * interest_rate * avg_loan_tenure
total_lost_interest = lost_interest_per_applicant * len(amber_good)

print(f"Average loan amount: ₹{avg_loan:,.0f}")
print(f"Lost interest per applicant: ₹{lost_interest_per_applicant:,.0f}")
print(f"Total lost interest income: ₹{total_lost_interest:,.0f}")

Average loan amount: ₹619,059
Lost interest per applicant: ₹278,577
Total lost interest income: ₹159,902,973


In [10]:
# Export data for Tableau
# 1. Risk band summary already saved

# 2. Full results CSV
df_results.to_csv(r'C:\Users\riyas\OneDrive\Desktop\Credit_Risk_project\full_results.csv', index=False)

# 3. Cost threshold data
thresholds = np.arange(0.1, 0.9, 0.01)
total_costs = []
approval_rates = []

COST_FN = 80000
COST_FP = 18000

from sklearn.metrics import confusion_matrix

for thresh in thresholds:
    preds = (lr_probs >= thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
    total_cost = (fn * COST_FN) + (fp * COST_FP)
    total_costs.append(total_cost)
    approval_rates.append((tp + fp) / len(y_test))

cost_df = pd.DataFrame({
    'threshold': thresholds,
    'total_cost': total_costs,
    'approval_rate': approval_rates
})
cost_df.to_csv(r'C:\Users\riyas\OneDrive\Desktop\Credit_Risk_project\cost_threshold_data.csv', index=False)

# 4. Feature importance CSV
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train_sm, y_train_sm)

feat_imp = pd.DataFrame({
    'feature': features,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

feat_imp.to_csv(r'C:\Users\riyas\OneDrive\Desktop\Credit_Risk_project\feature_importance.csv', index=False)

print("All files saved.")

All files saved.
